# Cat Qubit Loss Function

This notebook defines a loss function for **online optimization of cat qubit stabilization**, structured to mirror the `pi_pulse_loss_func` pattern in `challenge/1-challenge.ipynb`.

**System:** Storage mode $a$ coupled to a lossy buffer $b$ via:
$$H/\hbar = g_2^* a^2 b^\dagger + g_2 (a^\dagger)^2 b - \epsilon_d b^\dagger - \epsilon_d^* b$$
$$L_b = \sqrt{\kappa_b}\, b, \quad L_a = \sqrt{\kappa_a}\, a$$

**Knobs (4 real parameters):** $x = [\text{Re}(g_2),\ \text{Im}(g_2),\ \text{Re}(\epsilon_d),\ \text{Im}(\epsilon_d)]$

**Objective:** Maximize both $T_X$ and $T_Z$ while achieving target bias $\eta = T_Z / T_X$.

**Proxy strategy (measurement-efficient):**
- $T_X$ proxy — start from $|{+x_L}\rangle = (|\alpha\rangle + |-\alpha\rangle)/\sqrt{2}$, measure parity $\langle P \rangle$ at fixed short time
- $T_Z$ proxy — start from $|{+z_L}\rangle = |\alpha\rangle$, measure $\langle Z_L \rangle$ at fixed longer time

Both proxies avoid exponential-decay fitting, keeping per-call cost to two `mesolve` runs.

In [2]:
!pip install dynamiqs

  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   -------------------------------------- - 2.6/2.7 MB 15.0 MB/s eta 0:00:01
   ---------------------------------------- 2.7/2.7 MB 14.3 MB/s  0:00:00
   ---------------------------------------- 0.0/57.9 MB ? eta -:--:--
   --- ------------------------------------ 5.2/57.9 MB 24.5 MB/s eta 0:00:03
   --------- ------------------------------ 13.1/57.9 MB 31.6 MB/s eta 0:00:02
   --------------- ------------------------ 22.3/57.9 MB 35.2 MB/s eta 0:00:02
   -------------------- ------------------- 29.9/57.9 MB 35.8 MB/s eta 0:00:01
   --------------------------- ------------ 39.3/57.9 MB 37.9 MB/s eta 0:00:01
   --------------------------------- ------ 48.2/57.9 MB 38.4 MB/s eta 0:00:01
   -------------------------------------- - 55.3/57.9 MB 37.9 MB/s eta 0:00:01
   ---------------------------------------  57.7/57.9 MB 38.3 MB/s eta 0:00:01
   ---------

    python-dotenv (>=0.13.*) ; extra == 'standard'
                   ~~~~~~~^


In [ ]:
%pip install cmaes
import dynamiqs as dq
import jax.numpy as jnp
from jax import vmap, jit
from matplotlib import pyplot as plt
from cmaes import SepCMA

ModuleNotFoundError: No module named 'cmaes'

## Fixed System Parameters

These are hardware constants — **not** tunable by the online optimizer.

In [ ]:
# Hilbert space dimensions
NA = 15  # storage mode truncation
NB = 5   # buffer mode truncation

# Fixed loss rates [MHz]
KAPPA_B = 10.0  # fast buffer decay (enables adiabatic elimination)
KAPPA_A = 1.0   # single-photon storage loss

# Evaluation times for proxies [us]
# T_X is short (driven by single-photon loss ~ 1/kappa_a)
# T_Z is long (exponentially protected)
T_X_EVAL = 1.0
T_Z_EVAL = 50.0

# Loss weights
LAMBDA_X    = 1.0   # weight for T_X proxy term
LAMBDA_Z    = 0.5   # weight for T_Z proxy term
LAMBDA_BIAS = 0.1   # penalty weight for bias mismatch
ETA_TARGET  = 50.0  # target bias T_Z / T_X

## Loss Function

Structured analogously to `pi_pulse_loss_func` in the challenge notebook:
- Takes a 1D array of knob values
- Returns a scalar loss (lower = better)
- JIT-compiled and vmappable for batched CMA-ES evaluation

In [ ]:
@jit
def cat_loss_func(x):
    """
    Cat qubit stabilization loss function.

    Parameters
    ----------
    x : array, shape (4,)
        [Re(g2), Im(g2), Re(eps_d), Im(eps_d)]

    Returns
    -------
    float
        Scalar loss. Minimize this to maximize T_X, T_Z and hit target bias.
    """
    g_2   = x[0] + 1j * x[1]   # two-photon coupling [MHz]
    eps_d = x[2] + 1j * x[3]   # buffer drive amplitude [MHz]

    # ---- Build operators ------------------------------------------------
    a = dq.tensor(dq.destroy(NA), dq.eye(NB))   # storage annihilation
    b = dq.tensor(dq.eye(NA), dq.destroy(NB))   # buffer annihilation

    H = (jnp.conj(g_2) * a @ a @ b.dag()
         + g_2 * a.dag() @ a.dag() @ b
         - eps_d * b.dag()
         - jnp.conj(eps_d) * b)

    jump_ops = [jnp.sqrt(KAPPA_B) * b,
                jnp.sqrt(KAPPA_A) * a]

    # ---- Derived cat size (adiabatic-elimination estimate) --------------
    kappa_2   = 4.0 * jnp.abs(g_2)**2 / KAPPA_B
    eps_2     = 2.0 * g_2 * eps_d / KAPPA_B
    alpha_est = jnp.sqrt(jnp.abs(jnp.real(2.0 / kappa_2 * (eps_2 - KAPPA_A / 4.0))))

    # ---- Logical basis (storage tensor buffer vacuum) -------------------
    cat_pos = dq.tensor(dq.coherent(NA,  alpha_est), dq.fock(NB, 0))  # |+z_L> = |+alpha>
    cat_neg = dq.tensor(dq.coherent(NA, -alpha_est), dq.fock(NB, 0))  # |-z_L> = |-alpha>
    psi_x   = (cat_pos + cat_neg) / jnp.sqrt(2)                       # |+x_L> (even cat)

    # ---- Observable: parity P = exp(i pi a†a) ~ X_L -------------------
    # Does NOT require knowing alpha — robust to estimation error.
    parity = (1j * jnp.pi * (a.dag() @ a)).expm()

    # ---- Observable: Z_L = |+z_L><+z_L| - |-z_L><-z_L| ---------------
    sz_storage = (dq.coherent(NA,  alpha_est) @ dq.coherent(NA,  alpha_est).dag()
                - dq.coherent(NA, -alpha_est) @ dq.coherent(NA, -alpha_est).dag())
    z_logical  = dq.tensor(sz_storage, dq.eye(NB))

    # ---- T_X proxy: |+x_L> -> measure <P> at T_X_EVAL -----------------
    # <P> starts near +1; phase flips drive it toward 0.
    # High <P>(T_X_EVAL) => long T_X.
    res_x = dq.mesolve(
        H, jump_ops, psi_x,
        jnp.array([0.0, T_X_EVAL]),
        exp_ops=[parity],
        options=dq.Options(progress_meter=False)
    )
    sx_val = res_x.expects[0, -1].real   # <P> at t = T_X_EVAL

    # ---- T_Z proxy: |+z_L> -> measure <Z_L> at T_Z_EVAL ---------------
    # <Z_L> starts near +1; bit flips drive it toward 0.
    # High <Z_L>(T_Z_EVAL) => long T_Z.
    res_z = dq.mesolve(
        H, jump_ops, cat_pos,
        jnp.array([0.0, T_Z_EVAL]),
        exp_ops=[z_logical],
        options=dq.Options(progress_meter=False)
    )
    sz_val = res_z.expects[0, -1].real   # <Z_L> at t = T_Z_EVAL

    # ---- Bias proxy and penalty ----------------------------------------
    # Rough proxy: bias ~ sz_val / (1 - sx_val + eps)
    # A simpler penalty just penalises imbalance between the two proxies.
    bias_proxy  = sz_val / (jnp.abs(sx_val) + 1e-6)
    bias_penalty = (bias_proxy - ETA_TARGET)**2

    # ---- Combined loss (to be minimised) --------------------------------
    loss = (-LAMBDA_X * sx_val
            - LAMBDA_Z * sz_val
            + LAMBDA_BIAS * bias_penalty)
    return loss


# Batched version for CMA-ES population evaluation
batched_cat_loss_func = jit(vmap(cat_loss_func))

## Sanity Check

Evaluate at the nominal parameters used throughout the challenge notebook
($g_2 = 1$, $\epsilon_d = 4$, both real).

In [ ]:
# x = [Re(g2), Im(g2), Re(eps_d), Im(eps_d)]
x_nominal = jnp.array([1.0, 0.0, 4.0, 0.0])
x_bad     = jnp.array([0.1, 0.0, 1.0, 0.0])   # small drive -> tiny cat -> poor protection

print("Loss at nominal params: ", cat_loss_func(x_nominal))
print("Loss at weak params:    ", cat_loss_func(x_bad))

# Batched evaluation
xs = jnp.stack([x_nominal, x_bad])
print("Batched losses:         ", batched_cat_loss_func(xs))

## CMA-ES Optimization

Same CMA-ES loop structure as in `challenge/1-challenge.ipynb`, now optimizing the four cat qubit knobs.

In [ ]:
BATCH_SIZE = 12
N_EPOCHS   = 60

# Start near nominal operating point
mean0  = jnp.array([1.0, 0.0, 4.0, 0.0])
sigma0 = 0.3

optimizer = SepCMA(
    mean=mean0,
    sigma=sigma0,
    bounds=jnp.array([
        [0.1,  5.0],   # Re(g2)
        [-2.0, 2.0],   # Im(g2)
        [0.5, 10.0],   # Re(eps_d)
        [-2.0, 2.0],   # Im(eps_d)
    ]),
    population_size=BATCH_SIZE,
    seed=0,
)

mean_history   = []
loss_history   = []
loss_std_history = []

for epoch in range(N_EPOCHS):
    # Sample population
    xs = jnp.array([optimizer.ask() for _ in range(optimizer.population_size)])

    # Evaluate (batched)
    losses = batched_cat_loss_func(xs)

    # Tell CMA-ES (expects list of (solution, value) sorted ascending)
    solutions = [(xs[i], float(losses[i])) for i in range(len(xs))]
    solutions.sort(key=lambda sv: sv[1])
    optimizer.tell(solutions)

    mean_history.append(optimizer.mean.copy())
    loss_history.append(float(losses.mean()))
    loss_std_history.append(float(losses.std()))

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | mean loss {loss_history[-1]:.4f} | mean params {optimizer.mean}")

mean_history     = jnp.array(mean_history)
loss_history     = jnp.array(loss_history)
loss_std_history = jnp.array(loss_std_history)

In [ ]:
epochs = jnp.arange(N_EPOCHS)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), dpi=150)

# Loss curve
ax = axes[0]
ax.plot(epochs, loss_history, label="Mean loss")
ax.fill_between(epochs,
                loss_history - loss_std_history,
                loss_history + loss_std_history,
                alpha=0.2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("CMA-ES: Loss vs Epoch")
ax.grid(True, alpha=0.3)

# Parameter convergence
ax = axes[1]
labels = [r"$\mathrm{Re}(g_2)$", r"$\mathrm{Im}(g_2)$",
          r"$\mathrm{Re}(\epsilon_d)$", r"$\mathrm{Im}(\epsilon_d)$"]
for k, lbl in enumerate(labels):
    ax.plot(epochs, mean_history[:, k], label=lbl)
ax.set_xlabel("Epoch")
ax.set_ylabel("Parameter value")
ax.set_title("CMA-ES: Parameter Convergence")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()